# Analyse descriptive : table terminal-jour (une ligne par terminal et par jour)

**Objectif.** Décrire en profondeur la table agrégée issue de la requête « fiche d'identité » : qualité, complétude du calendrier, distributions, corrélations, saisonnalité, variabilité entre terminaux et au sein d'un même terminal, réseaux étrangers, inaboutis, captures. On évalue aussi ce que vaut l'ancien indicateur `flag_atypique`, fondé sur des z-scores calculés sur l'ensemble du parc.

Aucun modèle. La question est : **qu'est-ce qui est normal pour un terminal, et à quoi ressemble ce qui ne l'est pas ?**

Ce notebook complète l'analyse de la table des opérations. Celle-ci montre ce que l'agrégation journalière **ne voit pas** : la dynamique dans la journée.

## 0. Imports et paramètres
Python 3.6, pandas 1.1.5, lecture par morceaux, conversion explicite des types.

In [ ]:
import time

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import matplotlib.pyplot as plt
    GRAPHIQUES = True
except ImportError:
    GRAPHIQUES = False

try:
    import dataiku
    SUR_DATAIKU = True
except ImportError:
    SUR_DATAIKU = False

# --- Source ---
NOM_DATASET_SOURCE = "NOM_DU_DATASET"            # une ligne = un terminal, un jour
CHEMIN_CSV_LOCAL = "data/exemple_terminal_jour.csv"   # utilisé seulement hors Dataiku
TAILLE_MORCEAU = 500000                              # lignes lues à la fois (baisser si le kernel manque de mémoire)

# --- Périmètre ---
DATE_DEBUT = None                  # ex. "2025-01-01" ; None = tout l'historique
DATE_FIN = None                    # ex. "2025-12-31"
GARDER_SEULEMENT_METROPOLE_ACTIFS = False   # True = hors DOM-TOM et statut ACTIF uniquement

# --- Analyse ---
MIN_JOURS_TERMINAL = 60                 # terminaux retenus pour les analyses « par terminal »

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
if GRAPHIQUES:
    plt.rcParams["figure.figsize"] = (13, 4.5)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3

INDICATEURS = {
    "ret_nb": "retraits",
    "ret_montant_total": "montant total",
    "ret_montant_moyen": "montant moyen",
    "ret_pct_nuit": "% retraits de nuit",
    "ret_nb_inaboutis": "retraits inaboutis",
    "taux_inaboutis_pct": "% inaboutis",
    "cap_nb": "captures",
    "cap_cos_nb": "captures (réseau interne)",
    "nb_ope_reseau_etranger_sur": "retraits cartes étrangères",
    "taux_reseau_etranger_sur_pct": "% cartes étrangères",
    "nb_ope_reseau_exotique": "retraits réseaux rares",
}
TEXTES = ["num_automate", "code_entite_de_gestion", "type_gab_e_i", "statut_gab", "code_postale_emplacement",
          "raisons_atypisme"]
NUMERIQUES = (["ret_nb_cos", "ret_nb_horscos", "ret_nb_jours_actifs", "ret_montant_min", "ret_montant_max",
               "flag_atypique"] + list(INDICATEURS))
COLONNES_UTILES = ["annee", "mois", "jours"] + TEXTES + NUMERIQUES
CHRONO = {}


def chrono(etape, debut):
    """Mémorise et affiche la durée d'une étape."""
    CHRONO[etape] = round(time.time() - debut, 1)
    print("[{}] {} s".format(etape, CHRONO[etape]))

## 1. Chargement par morceaux
On ne lit que les colonnes utiles, morceau par morceau. Chaque morceau est typé (nombres en `float32`, identifiants nettoyés) avant d'être gardé : la mémoire reste faible. Les colonnes absentes du dataset sont signalées puis créées vides, pour que la suite du notebook ne plante pas.

In [ ]:
def colonnes_disponibles():
    if SUR_DATAIKU:
        return [c["name"] for c in dataiku.Dataset(NOM_DATASET_SOURCE).read_schema()]
    return list(pd.read_csv(CHEMIN_CSV_LOCAL, nrows=0).columns)


def lire_morceaux(colonnes):
    """Donne les données morceau par morceau, depuis Dataiku ou un CSV local."""
    if SUR_DATAIKU:
        return dataiku.Dataset(NOM_DATASET_SOURCE).iter_dataframes(chunksize=TAILLE_MORCEAU, columns=colonnes)
    return pd.read_csv(CHEMIN_CSV_LOCAL, usecols=colonnes, chunksize=TAILLE_MORCEAU, low_memory=False)


def ajouter_date(m):
    """Date construite à partir d'annee, mois, jours (NaT si invalide)."""
    parties = pd.DataFrame({"year": pd.to_numeric(m["annee"], errors="coerce"),
                            "month": pd.to_numeric(m["mois"], errors="coerce"),
                            "day": pd.to_numeric(m["jours"], errors="coerce")})
    return m.assign(date=pd.to_datetime(parties, errors="coerce"))


def nettoyer_morceau(m):
    """Types légers, identifiants propres, périmètre éventuel."""
    m = ajouter_date(m)
    for col in TEXTES:
        if col in m.columns:
            m[col] = m[col].astype(str).str.strip().replace({"nan": np.nan, "None": np.nan, "": np.nan})
    if "code_postale_emplacement" in m.columns:
        m["code_postale_emplacement"] = m["code_postale_emplacement"].str.zfill(5)
    for col in NUMERIQUES:
        if col in m.columns:
            m[col] = pd.to_numeric(m[col], errors="coerce").astype("float32")
    if GARDER_SEULEMENT_METROPOLE_ACTIFS:
        cp = m["code_postale_emplacement"].fillna("")
        m = m[(~cp.str.startswith(("97", "98"))) & (m["statut_gab"].str.upper() == "ACTIF")]
    if DATE_DEBUT:
        m = m[m["date"] >= pd.Timestamp(DATE_DEBUT)]
    if DATE_FIN:
        m = m[m["date"] <= pd.Timestamp(DATE_FIN)]
    return m.drop(columns=["annee", "mois", "jours"])


t0 = time.time()
dispo = colonnes_disponibles()
a_lire = [c for c in COLONNES_UTILES if c in dispo]
manquantes = [c for c in COLONNES_UTILES if c not in dispo]
print("Colonnes du dataset :", len(dispo))
print("Colonnes utiles absentes du dataset :", manquantes or "aucune")

morceaux, nb_lus, dates_invalides = [], 0, 0
for m in lire_morceaux(a_lire):
    nb_lus += len(m)
    m = nettoyer_morceau(m)
    dates_invalides += int(m["date"].isna().sum())
    morceaux.append(m)
    print("  {:,} lignes lues".format(nb_lus).replace(",", " "))
fiche = pd.concat(morceaux, ignore_index=True)
del morceaux
for col in manquantes:
    if col not in ("annee", "mois", "jours"):
        fiche[col] = np.nan
fiche = fiche[fiche["date"].notna()]
fiche["jour_semaine"] = fiche["date"].dt.dayofweek
fiche = fiche.sort_values(["num_automate", "date"]).reset_index(drop=True)
chrono("chargement", t0)
print("Lignes : {:,} | terminaux : {:,} | du {} au {} | dates invalides écartées : {:,}".format(
    len(fiche), fiche["num_automate"].nunique(), fiche["date"].min().date(), fiche["date"].max().date(),
    dates_invalides).replace(",", " "))
print("Mémoire du tableau : {:.2f} Go".format(fiche.memory_usage(deep=True).sum() / 1e9))

## 2. Qualité et cohérence

In [ ]:
q = pd.DataFrame({"nb_null": fiche.isna().sum(), "pct_null": (100 * fiche.isna().mean()).round(3),
                  "nb_valeurs_distinctes": fiche.nunique()})
display(q.sort_values("pct_null", ascending=False))

doublons = fiche.duplicated(["num_automate", "date"]).sum()
print("Doublons (terminal, date) :", doublons)
print("Dates invalides :", fiche["date"].isna().sum())

controles = {
    "ret_nb != ret_nb_cos + ret_nb_horscos": (fiche["ret_nb"] != fiche["ret_nb_cos"] + fiche["ret_nb_horscos"]),
    "ret_nb_jours_actifs différent de 1": (fiche["ret_nb_jours_actifs"] != 1),
    "montant min > montant moyen": (fiche["ret_montant_min"] > fiche["ret_montant_moyen"] + 1e-6),
    "montant moyen > montant max": (fiche["ret_montant_moyen"] > fiche["ret_montant_max"] + 1e-6),
    "inaboutis > retraits": (fiche["ret_nb_inaboutis"] > fiche["ret_nb"]),
    "taux en dehors de [0, 100]": ~fiche["taux_inaboutis_pct"].fillna(0).between(0, 100),
    "journées à 0 retrait (captures seules)": (fiche["ret_nb"] == 0),
}
display(pd.DataFrame({"nb_lignes": {k: int(v.sum()) for k, v in controles.items()},
                      "pct": {k: round(100 * v.mean(), 3) for k, v in controles.items()}}))
for c in ["type_gab_e_i", "statut_gab", "code_entite_de_gestion"]:
    print("\n" + c)
    print(fiche.drop_duplicates("num_automate")[c].value_counts(dropna=False).head(15).to_string())

## 3. Complétude du calendrier
La table ne contient une ligne que si le terminal a eu au moins une opération ce jour-là. **Un terminal en panne toute une journée est donc invisible** : la ligne manque, elle ne vaut pas zéro. On mesure ces jours manquants.

In [ ]:
vie = fiche.groupby("num_automate")["date"].agg(["min", "max", "count"])
vie["duree"] = (vie["max"] - vie["min"]).dt.days + 1
vie["jours_manquants"] = vie["duree"] - vie["count"]
vie["pct_manquants"] = 100 * vie["jours_manquants"] / vie["duree"]
print("Journées manquantes à l'intérieur des durées de vie : {:,} ({:.2f} % du calendrier attendu)".format(
    int(vie["jours_manquants"].sum()), 100 * vie["jours_manquants"].sum() / vie["duree"].sum()).replace(",", " "))
display(vie["pct_manquants"].describe(percentiles=[.5, .75, .9, .95, .99]).round(2).to_frame().T)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].hist(vie["pct_manquants"], bins=60)
axes[0].set_yscale("log")
axes[0].set_title("% de jours manquants par terminal")
parc = fiche.groupby("date")["num_automate"].nunique()
axes[1].plot(parc.index, parc.values, lw=0.8)
axes[1].set_title("terminaux présents dans la table, par jour")
plt.tight_layout()
plt.show()

# Longueur des trous (suites de jours manquants) : pannes longues ou absences ponctuelles ?
ecart = fiche.groupby("num_automate")["date"].diff().dt.days
trous = (ecart[ecart > 1] - 1).clip(upper=60)
print("Longueur des trous (jours consécutifs manquants, tronquée à 60) :")
display(trous.value_counts().sort_index().to_frame("nb_trous").T)

## 4. Distributions des indicateurs
Ce qu'on cherche : asymétrie, part de zéros, valeurs extrêmes. Ces propriétés conditionnent le choix des statistiques : médiane plutôt que moyenne, transformation log, traitement des zéros.

In [ ]:
cols = list(INDICATEURS)
desc = fiche[cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99, .999]).T
desc["pct_zeros"] = (100 * (fiche[cols] == 0).mean()).round(2)
desc["asymetrie"] = fiche[cols].skew().round(2)
desc.index = [INDICATEURS[c] for c in desc.index]
display(desc.round(2))

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
for ax, c in zip(axes.ravel(), cols):
    v = fiche[c].dropna()
    ax.hist(np.log1p(v.clip(lower=0)), bins=60)
    ax.set_title(INDICATEURS[c] + " (log1p)", fontsize=9)
for ax in axes.ravel()[len(cols):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Corrélations entre indicateurs
Corrélation de Spearman (rangs), robuste aux valeurs extrêmes. Les paires très corrélées sont redondantes : en garder une seule évite de compter deux fois la même information.

In [ ]:
ech = fiche[cols].sample(min(len(fiche), 500000), random_state=0)
corr = ech.rank().corr()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)))
ax.set_xticklabels([INDICATEURS[c] for c in cols], rotation=90, fontsize=8)
ax.set_yticks(range(len(cols)))
ax.set_yticklabels([INDICATEURS[c] for c in cols], fontsize=8)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, "{:.2f}".format(corr.values[i, j]), ha="center", va="center", fontsize=6)
plt.colorbar(im)
plt.title("Corrélations de Spearman")
plt.show()
fortes = corr.where(np.triu(np.ones(corr.shape, dtype=bool), 1)).stack()
fortes = fortes.reindex(fortes.abs().sort_values(ascending=False).index).head(10)
print("Paires les plus corrélées :")
display(fortes.rename(lambda c: INDICATEURS.get(c, c), level=0).rename(lambda c: INDICATEURS.get(c, c), level=1)
        .round(3).to_frame("spearman"))

## 6. Saisonnalité et tendance du parc

In [ ]:
total = fiche.groupby("date")[["ret_nb", "ret_nb_inaboutis", "cap_nb", "nb_ope_reseau_etranger_sur"]].sum()
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(total.index, total["ret_nb"], lw=0.5, alpha=0.5)
axes[0].plot(total.index, total["ret_nb"].rolling(28, center=True).mean(), lw=1.8)
axes[0].set_title("Retraits par jour, parc complet (et moyenne 28 jours)")
axes[1].plot(total.index, total["nb_ope_reseau_etranger_sur"].rolling(7, center=True).mean(), color="tab:purple")
axes[1].set_title("Retraits cartes étrangères par jour (moyenne 7 jours) : saisonnalité touristique")
plt.tight_layout()
plt.show()

par_terminal_jour = fiche.groupby([fiche["date"].dt.month, "jour_semaine"])["ret_nb"].mean().unstack()
par_terminal_jour.columns = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(par_terminal_jour.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(7))
ax.set_xticklabels(par_terminal_jour.columns)
ax.set_yticks(range(len(par_terminal_jour)))
ax.set_yticklabels(par_terminal_jour.index)
ax.set_ylabel("mois")
ax.set_title("Retraits moyens par terminal et par jour : mois x jour de semaine")
plt.colorbar(im)
plt.show()

annuel = fiche.groupby([fiche["date"].dt.year, fiche["date"].dt.month])["ret_nb"].mean().unstack(0)
annuel.plot(marker="o", title="Retraits moyens par terminal et par jour, selon le mois (une courbe par année)")
plt.show()

## 7. Variabilité entre terminaux et au sein d'un même terminal
Pour chaque indicateur, la variance totale se décompose en une part **entre terminaux** (les terminaux sont différents les uns des autres) et une part **au sein du terminal** (un terminal varie d'un jour à l'autre).

Si la part entre terminaux domine, comparer un terminal au reste du parc mesure surtout sa **taille** ou son **type**, pas une anomalie. C'est l'argument pour comparer chaque terminal **à lui-même**.

In [ ]:
lignes = []
for c in cols:
    v = fiche[["num_automate", c]].dropna()
    moy_terminal = v.groupby("num_automate")[c].transform("mean")
    var_tot = v[c].var(ddof=0)
    var_entre = ((moy_terminal - v[c].mean()) ** 2).mean()
    lignes.append({"indicateur": INDICATEURS[c], "part_entre_terminaux_pct": 100 * var_entre / var_tot if var_tot else np.nan})
decomp = pd.DataFrame(lignes).set_index("indicateur").round(1)
decomp["part_au_sein_du_terminal_pct"] = (100 - decomp["part_entre_terminaux_pct"]).round(1)
display(decomp.sort_values("part_entre_terminaux_pct", ascending=False))

## 8. Régularité des terminaux
- **Coefficient de variation** (écart-type / moyenne) des retraits quotidiens : un terminal très irrégulier rend toute alerte fragile.
- **Autocorrélation à 1 et 7 jours** : si celle à 7 jours est plus forte, le terminal suit un rythme hebdomadaire, et il faut comparer un lundi aux lundis.

In [ ]:
f = fiche[["num_automate", "date", "ret_nb"]].copy()
g = f.groupby("num_automate")["ret_nb"]
f["z"] = (f["ret_nb"] - g.transform("mean")) / g.transform("std").replace(0, np.nan)
for k in (1, 7):
    decale = f.groupby("num_automate")[["date", "z"]].shift(k)
    valide = (f["date"] - decale["date"]).dt.days == k
    f["prod_{}".format(k)] = np.where(valide, f["z"] * decale["z"], np.nan)
stab = f.groupby("num_automate").agg(n=("ret_nb", "size"), moyenne=("ret_nb", "mean"), ecart_type=("ret_nb", "std"),
                                     autocorr_1j=("prod_1", "mean"), autocorr_7j=("prod_7", "mean"))
stab = stab[stab["n"] >= MIN_JOURS_TERMINAL]
stab["cv"] = stab["ecart_type"] / stab["moyenne"].replace(0, np.nan)
display(stab[["cv", "autocorr_1j", "autocorr_7j"]].describe(percentiles=[.1, .25, .5, .75, .9]).round(3))
print("Terminaux dont le rythme hebdomadaire domine (autocorrélation 7 j > 1 j) : {:.1f} %".format(
    100 * (stab["autocorr_7j"] > stab["autocorr_1j"]).mean()))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].hist(stab["cv"].clip(upper=3), bins=60)
axes[0].set_title("Coefficient de variation des retraits quotidiens, par terminal")
axes[1].scatter(stab["autocorr_1j"], stab["autocorr_7j"], s=3, alpha=0.3)
axes[1].plot([-1, 1], [-1, 1], color="grey", lw=1)
axes[1].set_xlabel("autocorrélation à 1 jour")
axes[1].set_ylabel("autocorrélation à 7 jours")
axes[1].set_title("Rythme quotidien ou hebdomadaire ?")
plt.tight_layout()
plt.show()

## 9. Cartes étrangères

In [ ]:
etr = fiche.groupby("num_automate").agg(retraits=("ret_nb", "sum"), etrangers=("nb_ope_reseau_etranger_sur", "sum"),
                                        jours=("date", "size"))
etr = etr[etr["jours"] >= MIN_JOURS_TERMINAL]
etr["part_pct"] = 100 * etr["etrangers"] / etr["retraits"].replace(0, np.nan)
seuil_top = etr["part_pct"].quantile(0.95)
print("Part médiane de cartes étrangères par terminal : {:.3f} % ; 95e centile : {:.3f} %".format(etr["part_pct"].median(), seuil_top))
display(etr.nlargest(20, "part_pct").round(3))

top = etr.index[etr["part_pct"] >= seuil_top]
m = fiche.assign(groupe=np.where(fiche["num_automate"].isin(top), "5 % de terminaux les plus touristiques", "autres terminal"))
m = m.groupby(["groupe", m["date"].dt.to_period("M")])["taux_reseau_etranger_sur_pct"].mean().unstack(0)
m.index = m.index.to_timestamp()
m.plot(title="% moyen de cartes étrangères par mois", logy=True)
plt.show()

## 10. Inaboutis et captures : des événements rares
Ces indicateurs valent zéro la plupart du temps. On regarde la fréquence des journées non nulles, leur concentration sur quelques terminaux, et les journées extrêmes.

In [ ]:
for c in ["ret_nb_inaboutis", "cap_nb", "cap_cos_nb"]:
    v = fiche[c]
    par_terminal = fiche.groupby("num_automate")[c].sum().sort_values(ascending=False)
    k = max(1, int(len(par_terminal) * 0.05))
    print("{:<22} journées non nulles : {:6.2f} % | part portée par 5 % des terminaux : {:5.1f} % | max sur une journée : {}".format(
        INDICATEURS[c], 100 * (v > 0).mean(), 100 * par_terminal.head(k).sum() / max(par_terminal.sum(), 1), v.max()))
    print("   distribution des valeurs par journée :",
          v.clip(upper=10).value_counts().sort_index().to_dict())

print("\nJournées les plus extrêmes en inaboutis :")
display(fiche.nlargest(15, "ret_nb_inaboutis")[["num_automate", "date", "ret_nb", "ret_nb_inaboutis", "taux_inaboutis_pct"]])
print("Journées les plus extrêmes en captures :")
display(fiche.nlargest(15, "cap_nb")[["num_automate", "date", "ret_nb", "cap_nb", "cap_cos_nb"]])

## 11. Que vaut l'ancien `flag_atypique` ?
Il est déclenché quand au moins deux z-scores dépassent 2. Ces z-scores sont calculés par rapport à **l'ensemble du parc** du même mois. Si les mêmes terminal sont signalés tout le temps, le flag mesure le **profil** du terminal (grand, touristique…), pas une anomalie ponctuelle.

In [ ]:
if fiche["flag_atypique"].isna().all():
    print("Colonne flag_atypique absente du dataset : section ignorée.")
else:
    print("Part des journées signalées : {:.2f} %".format(100 * fiche["flag_atypique"].mean()))
    raisons = fiche["raisons_atypisme"].dropna().astype(str).str.split(", ").explode()
    raisons = raisons[raisons.str.len() > 0]
    if len(raisons):
        display(raisons.value_counts().to_frame("nb_journees"))

    taux_flag = fiche.groupby("num_automate").agg(taux_flag=("flag_atypique", "mean"), retraits_moyens=("ret_nb", "mean"),
                                                  jours=("date", "size"))
    taux_flag = taux_flag[taux_flag["jours"] >= MIN_JOURS_TERMINAL]
    k = max(1, int(len(taux_flag) * 0.05))
    print("Part des signalements portée par les 5 % de terminaux les plus signalés : {:.1f} %".format(
        100 * (taux_flag["taux_flag"] * taux_flag["jours"]).nlargest(k).sum() /
        max((taux_flag["taux_flag"] * taux_flag["jours"]).sum(), 1)))
    print("terminaux signalés plus d'un jour sur deux :", int((taux_flag["taux_flag"] > 0.5).sum()))
    print("Corrélation (Spearman) entre volume moyen du terminal et fréquence de signalement : {:.2f}".format(
        taux_flag[["taux_flag", "retraits_moyens"]].rank().corr().iloc[0, 1]))

## 12. Calendrier complet et habitude glissante de chaque terminal
**Calendrier complet.** On recrée toutes les journées de la vie de chaque terminal, de sa première à sa dernière opération. Une journée absente de la table devient une journée à **0 opération**, marquée `jour_absent`. Une panne d'une journée entière devient ainsi visible.

**Habitude glissante.** Pour chaque journée, l'habitude est la **médiane des 8 mêmes jours de semaine précédents** (un lundi est comparé aux 8 lundis d'avant), et la dispersion est la MAD sur ces mêmes 8 jours. Cette habitude suit les évolutions du terminal : tendance, changement d'emplacement, saison.

**Écart** = (valeur − habitude) / max(1,4826 × MAD ; √habitude). Le plancher √habitude correspond au bruit naturel d'un comptage : il évite qu'un terminal très régulier déclenche une alerte pour un retrait de plus.

In [ ]:
SEUIL_ECART = 4          # au-delà de 4 « écarts », la journée est considérée comme atypique
MIN_HABITUDE = 5         # une baisse n'est jugée que si le terminal fait habituellement au moins 5 retraits ce jour-là
INDICATEURS_SUIVIS = {"ret_nb": "retraits", "ret_nb_inaboutis": "inaboutis", "cap_nb": "captures",
                       "nb_ope_reseau_etranger_sur": "cartes étrangères"}


def calendrier_complet(df, colonnes):
    """Une ligne par terminal et par jour de sa durée de vie ; jours absents remplis à 0."""
    d = df.drop_duplicates(["num_automate", "date"])[["num_automate", "date"] + colonnes]
    vie = d.groupby("num_automate")["date"].agg(["min", "max"])
    n = ((vie["max"] - vie["min"]).dt.days + 1).values
    debut_bloc = np.repeat(np.cumsum(n) - n, n)
    decalage = np.arange(n.sum()) - debut_bloc
    cal = pd.DataFrame({"num_automate": np.repeat(vie.index.values, n),
                        "date": np.repeat(vie["min"].values, n) + decalage.astype("timedelta64[D]")})
    cal = cal.merge(d, on=["num_automate", "date"], how="left")
    cal["jour_absent"] = cal[colonnes[0]].isna()
    cal[colonnes] = cal[colonnes].fillna(0)
    cal["jour_semaine"] = cal["date"].dt.dayofweek
    return cal.sort_values(["num_automate", "date"]).reset_index(drop=True)


def ecart_habitude(cal, col, fenetre=8, min_obs=4):
    """Habitude glissante (médiane des 8 mêmes jours de semaine précédents), MAD et écart robuste."""
    cles = [cal["num_automate"], cal["jour_semaine"]]
    precedent = cal.groupby(cles)[col].shift(1)
    hab = (precedent.groupby(cles).rolling(fenetre, min_periods=min_obs).median()
           .reset_index(level=[0, 1], drop=True).sort_index())
    dev = (cal[col] - hab).abs()
    mad = (dev.groupby(cles).shift(1).groupby(cles).rolling(fenetre, min_periods=min_obs).median()
           .reset_index(level=[0, 1], drop=True).sort_index())
    echelle = np.maximum(1.4826 * mad.fillna(0), np.sqrt(hab.clip(lower=1)))
    return hab, (cal[col] - hab) / echelle


import time
t0 = time.time()
cal = calendrier_complet(fiche, list(INDICATEURS_SUIVIS))
for col in INDICATEURS_SUIVIS:
    cal["hab_" + col], cal["ecart_" + col] = ecart_habitude(cal, col)
print("Calendrier complet : {:,} journées, dont {:,} absentes de la table ({:.2f} %) — calculé en {:.0f} s".format(
    len(cal), int(cal["jour_absent"].sum()), 100 * cal["jour_absent"].mean(), time.time() - t0).replace(",", " "))


def qualifier(cal):
    """Type d'anomalie de chaque journée, à partir des écarts."""
    e = cal["ecart_ret_nb"]
    type_ = pd.Series("", index=cal.index)
    type_[(e < -SEUIL_ECART) & (cal["hab_ret_nb"] >= MIN_HABITUDE)] = "baisse"
    type_[cal["jour_absent"] & (cal["hab_ret_nb"] >= MIN_HABITUDE)] = "absence (aucune opération)"
    type_[e > SEUIL_ECART] = "hausse"
    for col, nom in list(INDICATEURS_SUIVIS.items())[1:]:
        pic = (cal["ecart_" + col] > SEUIL_ECART) & (cal[col] >= 3)
        type_[pic] = np.where(type_[pic] == "", "pic " + nom, type_[pic] + " + pic " + nom)
    return type_


cal["anomalie"] = qualifier(cal)
cal["est_anomalie"] = cal["anomalie"] != ""

## 13. Combien d'anomalies ? Le parc est-il calme ou agité ?
- Part des journées atypiques selon le seuil choisi : on vérifie que le seuil ne signale ni trop, ni trop peu.
- Nombre de terminaux atypiques **chaque jour** : un pic qui touche beaucoup de terminaux en même temps signale un événement commun (incident réseau, jour férié, grève), pas une anomalie de terminaux.
- Nombre de journées atypiques **par terminal** : quelques terminaux concentrent-ils les anomalies ?

In [ ]:
e = cal.loc[cal["hab_ret_nb"].notna(), "ecart_ret_nb"]
seuils = [2, 3, 4, 5, 6, 8]
courbe = pd.DataFrame({"seuil": seuils,
                       "% journées en baisse": [100 * (e < -s).mean() for s in seuils],
                       "% journées en hausse": [100 * (e > s).mean() for s in seuils]}).set_index("seuil")
display(courbe.round(3))
print("Types d'anomalies au seuil {} :".format(SEUIL_ECART))
display(cal.loc[cal["est_anomalie"], "anomalie"].value_counts().to_frame("journées"))

par_jour = cal.groupby("date").agg(terminal=("num_automate", "size"), anomalies=("est_anomalie", "sum"),
                                   absences=("jour_absent", "sum"))
par_jour["pct_terminaux_atypiques"] = 100 * par_jour["anomalies"] / par_jour["terminal"]
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
axes[0].plot(par_jour.index, par_jour["pct_terminaux_atypiques"], lw=0.8, color="tab:red")
axes[0].set_title("% de terminaux atypiques chaque jour (un pic = événement commun à beaucoup de terminaux)")
axes[1].plot(par_jour.index, par_jour["absences"], lw=0.8, color="black")
axes[1].set_title("Nombre de terminaux sans aucune opération dans la journée")
plt.tight_layout()
plt.show()
print("Jours où le plus de terminaux sont atypiques en même temps :")
display(par_jour.nlargest(15, "pct_terminaux_atypiques").round(2))

par_terminal_anom = cal.groupby("num_automate")["est_anomalie"].agg(["sum", "mean"])
par_terminal_anom.columns = ["journees_atypiques", "part_atypique"]
fig, ax = plt.subplots(figsize=(13, 4))
ax.hist(100 * par_terminal_anom["part_atypique"], bins=60)
ax.set_title("% de journées atypiques, par terminal")
plt.show()
k = max(1, int(len(par_terminal_anom) * 0.05))
print("Les 5 % de terminaux les plus atypiques portent {:.1f} % des journées atypiques".format(
    100 * par_terminal_anom["journees_atypiques"].nlargest(k).sum() / max(par_terminal_anom["journees_atypiques"].sum(), 1)))

## 14. Carte du parc : tous les terminaux, tous les jours, sur une seule image
Chaque ligne est un terminal (trié du plus actif au moins actif), chaque colonne un jour. **Bleu** : en dessous de l'habitude ; **rouge** : au-dessus ; **noir** : aucune opération ; **blanc** : normal.

Comment lire :
- **trait vertical** : beaucoup de terminaux touchés le même jour, donc un événement commun ;
- **trait horizontal** : un terminal durablement anormal (panne longue, déménagement, changement de niveau) ;
- **point isolé** : une anomalie ponctuelle d'un terminal, ce que l'on cherche à détecter.

In [ ]:
ANNEE_CARTE = None          # ex. 2025 ; None = toute la période
MAX_TERMINAUX_CARTE = 3000        # au-delà, on prend les terminaux les plus actifs (l'image reste lisible)


def carte_parc(cal, annee=None, max_terminaux=MAX_TERMINAUX_CARTE, entite=None):
    c = cal if annee is None else cal[cal["date"].dt.year == annee]
    if entite is not None:
        terminaux_entite = fiche.loc[fiche["code_entite_de_gestion"] == entite, "num_automate"].unique()
        c = c[c["num_automate"].isin(terminaux_entite)]
    niveau = c.groupby("num_automate")["ret_nb"].median().sort_values(ascending=False)
    terminaux = niveau.index[:max_terminaux]
    c = c[c["num_automate"].isin(terminaux)]
    val = c["ecart_ret_nb"].clip(-8, 8).where(~(c["jour_absent"] & (c["hab_ret_nb"] >= MIN_HABITUDE)), -99)
    m = pd.DataFrame({"g": c["num_automate"], "d": c["date"], "v": val}).pivot(index="g", columns="d", values="v")
    m = m.reindex(terminaux)
    cmap = plt.get_cmap("RdBu_r").copy() if hasattr(plt.get_cmap("RdBu_r"), "copy") else plt.get_cmap("RdBu_r")
    cmap.set_under("black")
    cmap.set_bad("#e6e6e6")
    fig, ax = plt.subplots(figsize=(16, 9))
    im = ax.imshow(np.ma.masked_invalid(m.values.astype(float)), aspect="auto", cmap=cmap, vmin=-8, vmax=8,
                   interpolation="nearest")
    jours = m.columns
    pos = [i for i, d in enumerate(jours) if d.day == 1]
    ax.set_xticks(pos)
    ax.set_xticklabels([jours[i].strftime("%m/%y") for i in pos], rotation=90, fontsize=8)
    ax.set_ylabel("terminal, du plus actif (haut) au moins actif (bas)")
    ax.set_title("Écart à l'habitude : {} terminal x {} jours{}".format(len(terminaux), len(jours),
                                                                  "" if annee is None else " — " + str(annee)))
    plt.colorbar(im, label="écart (bleu = baisse, rouge = hausse, noir = aucune opération)")
    plt.show()


carte_parc(cal, ANNEE_CARTE)

## 15. Profils de comportement des terminaux
Une typologie **descriptive**, par règles simples et lisibles, pour savoir à quoi ressemble chaque terminal sur la durée :

| Profil | Règle |
|---|---|
| Intermittent | plus de 20 % de jours sans aucune opération |
| Instable | plus de 10 % de journées atypiques |
| En déclin / en hausse | niveau des 90 derniers jours inférieur de 30 % / supérieur de 30 % à celui des 90 premiers |
| Saisonnier | mois le plus fort au moins 2 fois au-dessus du mois le plus faible |
| Stable (plat) | aucun des cas précédents |

Les règles sont appliquées dans cet ordre. Les seuils sont réglables ci-dessous.

In [ ]:
S_INTERMITTENT, S_INSTABLE, S_TENDANCE, S_SAISON = 0.20, 0.10, 0.30, 2.0

actifs = cal[~cal["jour_absent"]]
prof = pd.DataFrame({
    "jours": cal.groupby("num_automate").size(),
    "niveau_median": actifs.groupby("num_automate")["ret_nb"].median(),
    "part_absents": cal.groupby("num_automate")["jour_absent"].mean(),
    "part_atypiques": cal.groupby("num_automate")["est_anomalie"].mean(),
})
debut = cal.groupby("num_automate").head(90).groupby("num_automate")["ret_nb"].median()
fin = cal.groupby("num_automate").tail(90).groupby("num_automate")["ret_nb"].median()
prof["tendance"] = fin / debut.replace(0, np.nan) - 1
mensuel = actifs.groupby(["num_automate", actifs["date"].dt.month])["ret_nb"].median().unstack()
prof["amplitude_saison"] = mensuel.max(axis=1) / mensuel.min(axis=1).clip(lower=1)
prof = prof[prof["jours"] >= MIN_JOURS_TERMINAL]

prof["profil"] = "Stable (plat)"
prof.loc[prof["amplitude_saison"] >= S_SAISON, "profil"] = "Saisonnier"
prof.loc[(prof["jours"] >= 365) & (prof["tendance"] >= S_TENDANCE), "profil"] = "En hausse"
prof.loc[(prof["jours"] >= 365) & (prof["tendance"] <= -S_TENDANCE), "profil"] = "En déclin"
prof.loc[prof["part_atypiques"] > S_INSTABLE, "profil"] = "Instable"
prof.loc[prof["part_absents"] > S_INTERMITTENT, "profil"] = "Intermittent"
display(prof["profil"].value_counts().to_frame("terminal"))
display(prof.groupby("profil")[["niveau_median", "part_absents", "part_atypiques", "tendance", "amplitude_saison"]]
        .median().round(3))

In [ ]:
# Exemples de chaque profil : retraits lissés sur 7 jours, ramenés à la médiane du terminal
profils = prof["profil"].value_counts().index.tolist()
fig, axes = plt.subplots(len(profils), 4, figsize=(18, 2.6 * len(profils)), sharex=True, squeeze=False)
rng = np.random.RandomState(0)
for i, p in enumerate(profils):
    choix = prof.index[prof["profil"] == p]
    choix = rng.choice(choix, size=min(4, len(choix)), replace=False)
    for j in range(4):
        ax = axes[i, j]
        if j >= len(choix):
            ax.axis("off")
            continue
        s = cal[cal["num_automate"] == choix[j]].set_index("date")["ret_nb"]
        ax.plot(s.index, s.rolling(7, min_periods=1).mean() / max(s.median(), 1), lw=0.9)
        ax.axhline(1, color="grey", lw=0.6)
        ax.set_title("{} — {}".format(p, choix[j]), fontsize=9)
plt.tight_layout()
plt.show()

## 16. Fiche visuelle d'un terminal
Tout ce qu'il faut pour juger un terminal d'un coup d'œil :
1. ses retraits jour par jour, avec la **bande de son habitude** ; les journées atypiques sont marquées ;
2. son **calendrier** (une case par jour, colorée selon l'écart à l'habitude) ;
3. ses inaboutis, captures et cartes étrangères ;
4. son rythme de la semaine et de l'année.

In [ ]:
def fiche_terminal(num_automate, debut=None, fin=None):
    c = cal[cal["num_automate"] == num_automate].set_index("date")
    if debut:
        c = c[c.index >= pd.Timestamp(debut)]
    if fin:
        c = c[c.index <= pd.Timestamp(fin)]
    if c.empty:
        print("Terminal introuvable sur cette période.")
        return
    infos = fiche.loc[fiche["num_automate"] == num_automate].iloc[0]
    profil = prof["profil"].get(num_automate, "historique trop court")
    fig = plt.figure(figsize=(17, 13))
    grille = fig.add_gridspec(4, 3, height_ratios=[2.2, 1.6, 1.2, 1.3])

    ax = fig.add_subplot(grille[0, :])
    bande = 3 * np.sqrt(c["hab_ret_nb"].clip(lower=1))
    ax.fill_between(c.index, (c["hab_ret_nb"] - bande).clip(lower=0), c["hab_ret_nb"] + bande,
                    color="tab:blue", alpha=0.15, label="habitude ± bruit normal")
    ax.plot(c.index, c["hab_ret_nb"], color="tab:blue", lw=0.8, label="habitude (8 mêmes jours précédents)")
    ax.plot(c.index, c["ret_nb"], color="black", lw=0.6, label="retraits du jour")
    for typ, couleur in [("baisse", "tab:blue"), ("hausse", "tab:red"), ("absence", "black"), ("pic", "tab:orange")]:
        sel = c[c["anomalie"].str.contains(typ)]
        if len(sel):
            ax.scatter(sel.index, sel["ret_nb"], s=28, color=couleur, zorder=3, label=typ)
    ax.set_title("{} — {} | {} | {} | profil : {}".format(num_automate, infos.get("code_entite_de_gestion", ""),
                                                         infos.get("type_gab_e_i", ""), infos.get("statut_gab", ""),
                                                         profil))
    ax.legend(fontsize=8, ncol=4, loc="upper left")

    ax = fig.add_subplot(grille[1, :])
    semaines = ((c.index - c.index[0]).days + c.index[0].dayofweek) // 7
    val = c["ecart_ret_nb"].clip(-8, 8).where(~(c["jour_absent"] & (c["hab_ret_nb"] >= MIN_HABITUDE)), -99)
    m = np.full((7, semaines.max() + 1), np.nan)
    m[c.index.dayofweek, semaines] = val.values
    cmap = plt.get_cmap("RdBu_r").copy() if hasattr(plt.get_cmap("RdBu_r"), "copy") else plt.get_cmap("RdBu_r")
    cmap.set_under("black")
    cmap.set_bad("#eeeeee")
    ax.imshow(np.ma.masked_invalid(m), aspect="auto", cmap=cmap, vmin=-8, vmax=8, interpolation="nearest")
    ax.set_yticks(range(7))
    ax.set_yticklabels(["lun", "mar", "mer", "jeu", "ven", "sam", "dim"], fontsize=8)
    debuts_mois = [i for i, d in enumerate(c.index) if d.day == 1]
    ax.set_xticks([semaines[i] for i in debuts_mois])
    ax.set_xticklabels([c.index[i].strftime("%m/%y") for i in debuts_mois], fontsize=7, rotation=90)
    ax.set_title("Calendrier : écart à l'habitude (bleu = baisse, rouge = hausse, noir = aucune opération)")

    ax = fig.add_subplot(grille[2, :], sharex=fig.axes[0])
    for col, couleur in [("ret_nb_inaboutis", "tab:orange"), ("cap_nb", "tab:purple"),
                         ("nb_ope_reseau_etranger_sur", "tab:green")]:
        ax.plot(c.index, c[col], lw=0.7, color=couleur, label=INDICATEURS_SUIVIS[col])
    ax.legend(fontsize=8, ncol=3, loc="upper left")
    ax.set_title("Inaboutis, captures, cartes cartes étrangères par jour")

    actifs_terminal = c[~c["jour_absent"]]
    ax = fig.add_subplot(grille[3, 0])
    actifs_terminal.boxplot(column="ret_nb", by="jour_semaine", ax=ax, showfliers=False)
    ax.set_xticklabels(["lun", "mar", "mer", "jeu", "ven", "sam", "dim"])
    ax.set_title("Retraits selon le jour de semaine")
    ax.set_xlabel("")
    ax = fig.add_subplot(grille[3, 1])
    actifs_terminal.groupby([actifs_terminal.index.year, actifs_terminal.index.month])["ret_nb"].median().unstack(0) \
        .plot(ax=ax, marker="o", ms=3)
    ax.set_title("Médiane mensuelle (une courbe par année)")
    ax = fig.add_subplot(grille[3, 2])
    ax.hist(c["ecart_ret_nb"].dropna().clip(-10, 10), bins=50)
    ax.axvline(-SEUIL_ECART, color="red", lw=0.8)
    ax.axvline(SEUIL_ECART, color="red", lw=0.8)
    ax.set_title("Distribution de ses écarts")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

    anom = c[c["est_anomalie"]][["jour_absent", "ret_nb", "hab_ret_nb", "ecart_ret_nb", "ret_nb_inaboutis",
                                 "cap_nb", "nb_ope_reseau_etranger_sur", "anomalie"]]
    print("{} journées atypiques sur {} ({:.1f} %)".format(len(anom), len(c), 100 * len(anom) / len(c)))
    display(anom.round(1).tail(30))


# Exemple : le terminal le plus atypique du parc ; remplacer par n'importe quel numéro
fiche_terminal(par_terminal_anom["part_atypique"].idxmax())

## 17. Galerie : les terminaux à regarder en priorité
Les 12 terminaux qui ont le plus de journées atypiques (en nombre). Pour chacun : retraits par jour, habitude, journées atypiques en rouge. On repère ainsi, en un coup d'œil, les terminaux et les jours à approfondir avec `fiche_terminal`.

In [ ]:
def galerie(liste_terminaux, titre):
    n = len(liste_terminaux)
    lignes = int(np.ceil(n / 3))
    fig, axes = plt.subplots(lignes, 3, figsize=(18, 3 * lignes), squeeze=False)
    for ax, g in zip(axes.ravel(), liste_terminaux):
        c = cal[cal["num_automate"] == g].set_index("date")
        ax.plot(c.index, c["ret_nb"], lw=0.5, color="black")
        ax.plot(c.index, c["hab_ret_nb"], lw=0.8, color="tab:blue")
        a = c[c["est_anomalie"]]
        ax.scatter(a.index, a["ret_nb"], s=10, color="tab:red", zorder=3)
        ax.set_title("{} — {} journées atypiques".format(g, len(a)), fontsize=9)
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    plt.suptitle(titre, y=1.0)
    plt.tight_layout()
    plt.show()


galerie(par_terminal_anom["journees_atypiques"].nlargest(12).index.tolist(), "Terminaux avec le plus de journées atypiques")
galerie(list(np.random.RandomState(1).choice(par_terminal_anom.index, size=min(6, len(par_terminal_anom)), replace=False)),
        "Six terminaux pris au hasard, pour comparaison")

## 18. Changements de niveau durables (ruptures)
Une anomalie n'est pas toujours ponctuelle : un terminal peut changer de niveau d'un coup et y rester (travaux, fermeture d'un commerce voisin, déplacement). On compare, pour chaque jour, la médiane des 28 jours qui suivent à celle des 28 jours qui précèdent. Les plus forts changements sont listés.

In [ ]:
def ruptures(cal, fenetre=28, niveau_min=10):
    g = cal.groupby("num_automate")["ret_nb"]
    avant = g.transform(lambda s: s.rolling(fenetre, min_periods=fenetre).median().shift(1))
    apres = g.transform(lambda s: s[::-1].rolling(fenetre, min_periods=fenetre).median()[::-1])
    ratio = (apres + 1) / (avant + 1)
    r = cal[["num_automate", "date"]].assign(avant=avant, apres=apres, ratio=ratio)
    r = r[(r[["avant", "apres"]].max(axis=1) >= niveau_min) & r["ratio"].notna()]
    r["force"] = np.abs(np.log(r["ratio"]))
    meilleur = r.loc[r.groupby("num_automate")["force"].idxmax()]
    return meilleur.sort_values("force", ascending=False)


rupt = ruptures(cal)
print("Terminaux dont le niveau a été au moins divisé ou multiplié par 2 d'un coup : {}".format(int((rupt["force"] > np.log(2)).sum())))
display(rupt.head(20).round(2))

## 19. Synthèse
À compléter après exécution :

| Question | Constat |
|---|---|
| Qualité : doublons, incohérences, jours manquants | |
| Indicateurs redondants (corrélation > 0,9) | |
| Part de variance entre terminaux / au sein du terminal | |
| Rythme hebdomadaire dominant ? | |
| Saisonnalité annuelle et touristique | |
| Combien de journées atypiques au seuil retenu ? Trop, ou trop peu ? | |
| Événements communs à beaucoup de terminaux (dates) | |
| Répartition des profils : stable, saisonnier, en déclin, instable, intermittent | |
| Types d'anomalies les plus fréquents : baisse, absence, hausse, pics | |
| Ruptures de niveau durables | |
| Biais de l'ancien `flag_atypique` | |
| terminal et journées à examiner au grain opération (notebook Spark) | |